<a href="https://colab.research.google.com/github/sokrypton/7.571/blob/main/L8/pca.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PCA — From Scratch

1. **Implement PCA from scratch** using only NumPy
2. **Visualize** 3D → 2D projection with Plotly
3. **Explained variance** — how many dimensions do we need?
4. **Compare** with scikit-learn
5. **Neural PCA** in JAX — learn PCA with gradient descent

---

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
import matplotlib.animation as animation
import plotly.graph_objs as go
import plotly.express as px
from sklearn.datasets import make_blobs

import plotly.graph_objs as go
from matplotlib.colors import ListedColormap
import string

colorway = ["#1f77b4", "#ff7f0e", "#2ca02c",
              "#d62728","#9467bd", "#8c564b",
              "#e377c2", "#7f7f7f", "#bcbd22",
              "#17becf"]

c10 = lambda x: ListedColormap(colorway[:x])

def make_2D_plot(values, groups=None, s=10, c=None, cmap=None):
  '''
  Given values and groups, returns a scatter plot colored by group
  '''
  if groups is None:
    groups = np.zeros(len(values),int)
    if c is None: c = np.arange(len(values))
    if cmap is None: cmap = "plasma"
  for group in np.unique(groups):
    idx = groups == group
    plt.scatter(values[idx, 0], values[idx, 1],
                label=group, s=s, c=c, cmap=cmap)
  if len(np.unique(groups)) > 1:
    plt.legend(bbox_to_anchor=(1, 0, 0.5, 1), loc="upper left",)
  plt.axis("equal")

def make_3D_plot(values, groups=None, s=3):
  if groups is None:
    groups = np.zeros(len(values),int)
    if values.shape[-1] > 3:
      c = values[:,3]
      c = (c - c.min()) / (c.max() - c.min())
    else:
      c = None
  else:
    c = None
  plot = []
  for group in np.unique(groups):
    idx = groups == group
    plot.append(go.Scatter3d(
        x=values[idx,0],
        y=values[idx,1],
        z=values[idx,2],
        name=str(group),
        mode='markers',
        marker=dict(size=s,color=c)))

  go.Figure(data=plot,layout=go.Layout(
      colorway=colorway, scene={"aspectmode":"data"})).show()


In [ ]:
# Generate 3D data with 5 clusters
X_3d, groups = make_blobs(n_samples=500, n_features=3, centers=5, random_state=42)

make_3D_plot(X_3d, groups)

---
## 1. PCA from Scratch (NumPy)

PCA in 4 steps:
1. Center the data
2. Compute covariance matrix
3. Eigendecomposition → sort by largest eigenvalue
4. Project onto top-k eigenvectors

In [ ]:
def pca(X, n_components):
  """
  PCA from scratch.
  Returns: projected data, components (W), mean, eigenvalues
  """
  # Step 1: Center the data
  X_mean = X.mean(axis=0)
  X_centered = X - X_mean

  # Step 2: Covariance matrix
  cov = (X_centered.T @ X_centered) / (len(X) - 1)

  # Step 3: Eigendecomposition (eigh returns sorted ascending, so we flip)
  eigenvalues, eigenvectors = np.linalg.eigh(cov)
  eigenvalues = eigenvalues[::-1]
  eigenvectors = eigenvectors[:, ::-1]

  # Step 4: Take top-k components and project
  W = eigenvectors[:, :n_components]  # (d, k)
  Z = X_centered @ W                  # (n, k)

  # Step 5: Optional (reconstruct data)
  X_reconstructed = Z @ W.T + X_mean

  return Z, W, X_reconstructed

Z, W, X_reconstructed = pca(X_3d, n_components=2)
print(f'3D → 2D: {X_3d.shape} → {Z.shape}')
print(f'W (components): {W.shape}')

---
## 2. Visualize the Projection

3D data projected down to 2D, and then reconstructed back to 3D.

In [ ]:
# 2D projection
make_2D_plot(Z, groups)

In [ ]:
# Compare original vs reconstructed in 3D
fig = go.Figure()
fig.add_trace(go.Scatter3d(
    x=X_3d[:, 0], y=X_3d[:, 1], z=X_3d[:, 2],
    mode='markers', marker=dict(size=2, color='gray', opacity=0.3),
    name='Original'))
fig.add_trace(go.Scatter3d(
    x=X_reconstructed[:, 0], y=X_reconstructed[:, 1], z=X_reconstructed[:, 2],
    mode='markers', marker=dict(size=2, color='red', opacity=0.5),
    name='Reconstructed (2 PCs)'))
fig.update_layout(title='Original vs Reconstructed (3D → 2D → 3D)',
                  scene=dict(aspectmode='data'))
fig.show()

In [ ]:
# Scatter: input vs recovered values

def dist(X):
  return np.sqrt(np.square(X[:,None] - X[None,:]).sum(-1))

plt.figure(figsize=(10, 5))
plt.subplot(1,2,1)
plt.scatter(X_3d, X_reconstructed, s=1, alpha=0.3)
plt.xlabel('Input data'); plt.ylabel('Recovered data')
plt.axis('equal')
plt.subplot(1,2,2)
plt.scatter(dist(X_3d), dist(X_reconstructed), s=1, alpha=0.3)
plt.xlabel('DM(Input data)'); plt.ylabel('DM(Recovered data)')
plt.axis('equal')
plt.tight_layout()
plt.show()

---
## 4. The sklearn Way

In [ ]:
from sklearn.decomposition import PCA

sk_pca = PCA(n_components=2).fit(X_3d)
Z_sk = sk_pca.transform(X_3d)

# Side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, proj, title in [
    (axes[0], Z, 'Our PCA'),
    (axes[1], Z_sk, 'sklearn PCA'),
]:
    for g in np.unique(groups):
        idx = groups == g
        ax.scatter(proj[idx, 0], proj[idx, 1], s=10, alpha=0.6, label=f'{g}')
    ax.set_xlabel('PC 1'); ax.set_ylabel('PC 2')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('equal')
plt.suptitle('From Scratch vs scikit-learn', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Explained Variance — How Many Dimensions?

Generate higher-dimensional data and see how much variance each PC captures.

In [ ]:
# 10D data
X_10d, groups_10d = make_blobs(n_samples=500, n_features=10, centers=5, random_state=42)


# Run PCA keeping all components
sk_pca = PCA(n_components=10).fit(X_10d)


explained = sk_pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(1, 11), explained, color='#2c3e50')
axes[0].set_xlabel('Principal Component'); axes[0].set_ylabel('Variance Ratio')
axes[0].set_title('Explained Variance per PC', fontweight='bold')
axes[0].set_xticks(range(1, 11))

axes[1].plot(range(1, 11), cumulative, 'o-', color='#2c3e50', linewidth=2, markersize=8)
axes[1].set_xlabel('Number of Components'); axes[1].set_ylabel('Cumulative Variance')
axes[1].set_title('Cumulative Explained Variance', fontweight='bold')
axes[1].set_xticks(range(1, 11));
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


---
## 5. 🧪 Bonus: Neural PCA in JAX

A linear autoencoder with a bottleneck learns PCA!

```
X_c = X - mean(X)
reconstruction = X_c @ W @ W.T    # W is (d, k)
loss = ||X_c - reconstruction||²
```

The optimal `W` spans the same subspace as the top-k eigenvectors.

In [ ]:
import jax
import jax.numpy as jnp
from jax import grad, jit

def autoencoder_loss(W, X_c):
    """Linear autoencoder: encode then decode, minimize reconstruction error."""
    reconstruction = X_c @ W @ W.T   # (n, d) @ (d, k) @ (k, d) → (n, d)
    return jnp.mean((X_c - reconstruction) ** 2)


def train_neural_pca(X_np, n_components=2, lr=0.001, steps=300, snapshot_every=5):
    X_mean = X_np.mean(axis=0)
    X_c = jnp.array(X_np - X_mean)
    d = X_np.shape[1]

    # Initialise W randomly — shape (d, n_components)
    key = jax.random.PRNGKey(0)
    W = jax.random.normal(key, (d, n_components)) * 0.1

    grad_fn = jit(grad(autoencoder_loss))
    loss_fn = jit(autoencoder_loss)

    history = []
    for step in range(steps):
        g = grad_fn(W, X_c)
        W = W - lr * g

        if step % snapshot_every == 0 or step == steps - 1:
            projected = np.array(X_c @ W)          # (n, k)
            reconstructed = np.array(X_c @ W @ W.T) + X_mean  # back to original space
            loss = float(loss_fn(W, X_c))
            history.append({
                'step': step,
                'projected': projected,
                'reconstructed': reconstructed,
                'loss': loss,
            })

    return history, X_mean


nn_history, nn_mean = train_neural_pca(X_3d, n_components=2, steps=300, snapshot_every=15)
print(f'Recorded {len(nn_history)} snapshots')
print(f'Final loss: {nn_history[-1]["loss"]:.4f}')

In [ ]:
# Animate: 2D projection learning + loss curve
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
losses = [h['loss'] for h in nn_history]
steps_list = [h['step'] for h in nn_history]

def animate_nn(frame_idx):
    h = nn_history[frame_idx]
    proj = h['projected']

    ax1.clear()
    for g in np.unique(groups):
        idx = groups == g
        ax1.scatter(proj[idx, 0], proj[idx, 1], s=10, alpha=0.5, color=COLORS[g])
    ax1.set_xlabel('Component 1'); ax1.set_ylabel('Component 2')
    ax1.set_title(f'Step {h["step"]}', fontsize=13, fontweight='bold')
    ax1.axis('equal')

    ax2.clear()
    ax2.plot(steps_list[:frame_idx + 1], losses[:frame_idx + 1],
             '-', color='#2c3e50', linewidth=2)
    ax2.scatter(steps_list[frame_idx], losses[frame_idx],
                color='#e74c3c', s=80, zorder=5)
    ax2.set_xlabel('Training Step'); ax2.set_ylabel('Loss')
    ax2.set_title('Reconstruction Loss', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, steps_list[-1])
    ax2.set_ylim(0, max(losses) * 1.1)
    ax2.grid(alpha=0.3)

anim = animation.FuncAnimation(fig, animate_nn,
                                frames=len(nn_history),
                                interval=120, repeat=True)
plt.close(fig)
HTML(anim.to_html5_video())

---
## 📝 Key Takeaways

| Concept | Detail |
|---|---|
| **Algorithm** | Center → covariance → eigendecomposition → project |
| **Reconstruction** | `X_c @ W @ W.T + mean` maps back to original space |
| **Explained variance** | eigenvalue_k / sum(eigenvalues) — tells you how many PCs you need |
| **Neural PCA** | A linear autoencoder with bottleneck learns the same subspace as PCA |
| **Key difference** | PCA finds exact eigenvectors; neural version finds the subspace (up to rotation) |

# 6. Bonus: Real Example

In [ ]:
!wget -qnc https://raw.githubusercontent.com/sokrypton/7.571/refs/heads/main/L8/example.fasta

In [ ]:
def parse_fasta(filename):
  '''function to parse fasta'''
  # create empty lists to append names/seqs
  names = []
  seqs = []
  # open file
  lines = open(filename, "r")
  # go through file, line by line
  for line in lines:
    # remove linebreak
    line = line.rstrip()
    # if the first character is ">"
    if line[0] == ">":
      # save name
      names.append(line[1:])
      # start empty string
      seqs.append("")
    else:
      # add to existing string
      seqs[-1] += line
  # close file
  lines.close()
  return names, seqs

def filt_seqs(names, seqs):

  # get query (first) sequence
  query_seq = seqs[0]

  # convert sequence into numpy array of characters
  query_array = np.array(list(query_seq))

  # check which characters are not "-"
  query_non_gap = query_array != "-"

  # the length of query
  query_length = sum(query_non_gap)

  # make a new list of names/sequences
  new_names = []
  new_seqs = []

  # for each name and sequence
  for name,seq in zip(names,seqs):

    # convert sequence into numpy array of characters
    seq_array = np.array(list(seq))

    # select only positions that are non-gap in query
    seq_array = seq_array[query_non_gap]

    # count number of gaps remaining in sequence
    seq_gap_count = sum(seq_array == "-")

    # if there are more than 25% gaps, ignore
    if seq_gap_count/query_length <= 0.25:
      new_names.append(name)
      new_seqs.append("".join(seq_array))

  return new_names, new_seqs

def mk_msa(seqs):
  '''one hot encode msa'''
  alphabet = list("ARNDCQEGHILKMFPSTWYV-")
  states = len(alphabet)

  alpha = np.array(alphabet, dtype='|S1').view(np.uint8)
  msa = np.array([list(s) for s in seqs], dtype='|S1').view(np.uint8)
  for n in range(states):
    msa[msa == alpha[n]] = n
  msa[msa > states] = states-1
  return np.eye(states)[msa]

In [ ]:
names, seqs = parse_fasta("example.fasta")

In [ ]:
new_names, new_seqs = filt_seqs(names,seqs)

In [ ]:
msa = mk_msa(new_seqs)

In [ ]:
# reshape MSA from (N,L,A) to (N,L*A)
N,L,A = msa.shape
X = msa[...,:20].reshape(N,-1)
pc = PCA(4).fit_transform(X)

In [ ]:
make_3D_plot(pc)